# XLM-RoBERTa-Large Bi-Encoder
## RFQ–Supplier Match Classification

**Improvements over v1:**
1. **Bi-encoder** — separate encoding for RFQ and Supplier (no truncation loss)
2. **Gradual unfreezing** — 3-phase training schedule
3. **Focal Loss** with label smoothing
4. **Structured features** — country match, type overlap, region compatibility
5. **Tuned hyperparameters** — lower LR, more epochs, stronger regularisation
6. **xlm-roberta-large** — 550 M parameters

In [2]:
!pip install -q transformers datasets accelerate scikit-learn

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os, warnings, shutil
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
from transformers import (AutoTokenizer, AutoModel,
                          TrainingArguments, Trainer,
                          EarlyStoppingCallback)
from torch.utils.data import Dataset as TorchDataset
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not found! Go to Runtime → Change runtime type → GPU")

device = torch.device("cuda")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [4]:
LABEL2ID = {"match": 0, "weak_match": 1, "related": 2, "no_match": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

MODEL_NAME = "xlm-roberta-large"
RFQ_MAX_LEN = 256
SUP_MAX_LEN = 384
NUM_STRUCTURED = 7

EUROPEAN = {
    "DE","FR","ES","IT","NL","BE","AT","PT","PL","CZ","SK","HU","RO",
    "BG","HR","SI","LT","LV","EE","FI","SE","DK","IE","GR","LU","MT",
    "CY","CH","NO","GB","UK",
}

TYPE_NORM = {
    "manufacturer": "production",
    "production": "production",
    "customerspecificmanufacturing": "production",
    "wholesaler": "wholesaler",
    "trader": "wholesaler",
    "service": "service",
    "serviceprovider": "service",
}


def clean_array_field(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    if s.startswith("{") and s.endswith("}"):
        s = s[1:-1]
    return s.replace(",", ", ")


def build_rfq_text(row):
    parts = [
        row["rfq_title"],
        str(row["rfq_description"])[:500],
        f"Delivery: {row['delivery_location']}",
        f"Qty: {row['quantity']}",
        f"Types: {row['rfq_supplier_types']}",
    ]
    return " | ".join(p for p in parts if str(p).strip())


def build_supplier_text(row):
    parts = [
        row["supplier_name"],
        f"{row['supplier_country']} {row['distribution_area']}".strip(),
        row["supplier_types"],
        str(row["products"])[:300],
        row["product_categories"],
        row["keywords"],
        str(row["supplier_description"])[:400],
    ]
    return " | ".join(p for p in parts if str(p).strip())


def _norm_types(s):
    toks = {t.strip().lower() for t in str(s).split(",") if t.strip()}
    return {TYPE_NORM.get(t, t) for t in toks}


def type_jaccard(a, b):
    sa, sb = _norm_types(a), _norm_types(b)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def region_compat(delivery, sup_country, dist_area):
    da = str(dist_area).lower().strip()
    if da in ("", "nan"):
        return 0.5
    if da == "international":
        return 1.0
    if da == "europe":
        return 1.0 if delivery.upper() in EUROPEAN else 0.0
    if da in ("national", "local"):
        return 1.0 if delivery.upper() == sup_country.upper() else 0.0
    return 0.5


def extract_features(row):
    return [
        float(str(row["delivery_location"]).upper()
              == str(row["supplier_country"]).upper()),
        region_compat(row["delivery_location"],
                      row["supplier_country"],
                      row["distribution_area"]),
        type_jaccard(row["rfq_supplier_types"], row["supplier_types"]),
        min(len(str(row["rfq_description"])) / 1000.0, 3.0),
        min(len(str(row["supplier_description"])) / 1000.0, 5.0),
        float(len(str(row["keywords"]).strip()) > 0),
        float(len(str(row["products"]).strip()) > 0),
    ]

In [19]:
df = pd.read_csv("/content/result_dataset_balanced_56k.csv", on_bad_lines='skip', engine='python')
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")

ARRAY_COLS = ["rfq_supplier_types", "supplier_types", "products",
              "product_categories", "keywords"]
TEXT_COLS  = [
    "rfq_title", "rfq_description", "delivery_location", "quantity",
    "rfq_supplier_types", "supplier_name", "supplier_country",
    "distribution_area", "supplier_description", "supplier_types",
    "products", "product_categories", "keywords",
]

for col in ARRAY_COLS:
    df[col] = df[col].apply(clean_array_field)
for col in TEXT_COLS:
    df[col] = df[col].fillna("").astype(str)

df["text_rfq"]      = df.apply(build_rfq_text, axis=1)
df["text_supplier"] = df.apply(build_supplier_text, axis=1)
df["label"]         = df["match_type"].map(LABEL2ID)
df["features"]      = df.apply(extract_features, axis=1)

print(f"RFQ text:      mean={df['text_rfq'].str.len().mean():.0f}  "
      f"max={df['text_rfq'].str.len().max()} chars")
print(f"Supplier text: mean={df['text_supplier'].str.len().mean():.0f}  "
      f"max={df['text_supplier'].str.len().max()} chars")
print(f"\nStructured feature sample: {df['features'].iloc[0]}")

Loaded 50,429 rows, 16 columns
RFQ text:      mean=471  max=798 chars
Supplier text: mean=4064  max=48414 chars

Structured feature sample: [0.0, 1.0, 0.5, 0.309, 1.41, 1.0, 1.0]


In [20]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tv_idx, te_idx = next(gss1.split(df, df["label"], groups=df["rfq_id"]))
df_tv   = df.iloc[tv_idx].reset_index(drop=True)
df_test = df.iloc[te_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=42)
tr_idx, va_idx = next(gss2.split(df_tv, df_tv["label"],
                                  groups=df_tv["rfq_id"]))
df_train = df_tv.iloc[tr_idx].reset_index(drop=True)
df_val   = df_tv.iloc[va_idx].reset_index(drop=True)

print(f"Train: {len(df_train):,}   Val: {len(df_val):,}   Test: {len(df_test):,}")
print(f"\nLabel distribution (train):")
print(df_train["label"].value_counts().sort_index())

Train: 34,730   Val: 5,284   Test: 10,415

Label distribution (train):
label
0    9777
1    5733
2    9724
3    9496
Name: count, dtype: int64


In [21]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def batch_tokenize(texts, max_len):
    return tokenizer(
        texts, truncation=True, max_length=max_len,
        padding=False, return_attention_mask=True,
    )


print("Tokenizing RFQ texts ...")
train_rfq = batch_tokenize(df_train["text_rfq"].tolist(), RFQ_MAX_LEN)
val_rfq   = batch_tokenize(df_val["text_rfq"].tolist(),   RFQ_MAX_LEN)
test_rfq  = batch_tokenize(df_test["text_rfq"].tolist(),  RFQ_MAX_LEN)

print("Tokenizing Supplier texts ...")
train_sup = batch_tokenize(df_train["text_supplier"].tolist(), SUP_MAX_LEN)
val_sup   = batch_tokenize(df_val["text_supplier"].tolist(),   SUP_MAX_LEN)
test_sup  = batch_tokenize(df_test["text_supplier"].tolist(),  SUP_MAX_LEN)

print(f"Sample RFQ tokens:      {len(train_rfq['input_ids'][0])}")
print(f"Sample Supplier tokens: {len(train_sup['input_ids'][0])}")


class BiEncoderDataset(TorchDataset):
    def __init__(self, rfq_enc, sup_enc, labels, features):
        self.rfq_ids  = rfq_enc["input_ids"]
        self.rfq_mask = rfq_enc["attention_mask"]
        self.sup_ids  = sup_enc["input_ids"]
        self.sup_mask = sup_enc["attention_mask"]
        self.labels   = labels
        self.features = features

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "rfq_input_ids":       self.rfq_ids[idx],
            "rfq_attention_mask":  self.rfq_mask[idx],
            "sup_input_ids":       self.sup_ids[idx],
            "sup_attention_mask":  self.sup_mask[idx],
            "structured_features": self.features[idx],
            "labels":              self.labels[idx],
        }


ds_train = BiEncoderDataset(train_rfq, train_sup,
                            df_train["label"].tolist(),
                            df_train["features"].tolist())
ds_val   = BiEncoderDataset(val_rfq, val_sup,
                            df_val["label"].tolist(),
                            df_val["features"].tolist())
ds_test  = BiEncoderDataset(test_rfq, test_sup,
                            df_test["label"].tolist(),
                            df_test["features"].tolist())

print(f"\nDatasets ready \u2014 "
      f"train: {len(ds_train)}, val: {len(ds_val)}, test: {len(ds_test)}")

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing RFQ texts ...
Tokenizing Supplier texts ...
Sample RFQ tokens:      118
Sample Supplier tokens: 384

Datasets ready — train: 34730, val: 5284, test: 10415


In [22]:
class FocalLoss(nn.Module):
    """Focal Loss with optional label smoothing."""
    def __init__(self, gamma=2.0, label_smoothing=0.1, num_classes=4):
        super().__init__()
        self.gamma = gamma
        self.ls    = label_smoothing
        self.nc    = num_classes

    def forward(self, logits, labels):
        with torch.no_grad():
            targets = torch.full_like(logits, self.ls / (self.nc - 1))
            targets.scatter_(1, labels.unsqueeze(1), 1.0 - self.ls)
        log_p = F.log_softmax(logits, dim=-1)
        p     = log_p.exp()
        loss  = -((1 - p) ** self.gamma) * targets * log_p
        return loss.sum(dim=-1).mean()


class BiEncoderCollator:
    """Pads RFQ and Supplier sequences independently per batch."""
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def _pad(self, seqs, pad_val):
        t = [torch.tensor(s, dtype=torch.long) for s in seqs]
        return nn.utils.rnn.pad_sequence(t, batch_first=True,
                                         padding_value=pad_val)

    def __call__(self, batch):
        return {
            "rfq_input_ids":
                self._pad([b["rfq_input_ids"] for b in batch], self.pad_id),
            "rfq_attention_mask":
                self._pad([b["rfq_attention_mask"] for b in batch], 0),
            "sup_input_ids":
                self._pad([b["sup_input_ids"] for b in batch], self.pad_id),
            "sup_attention_mask":
                self._pad([b["sup_attention_mask"] for b in batch], 0),
            "structured_features": torch.tensor(
                [b["structured_features"] for b in batch], dtype=torch.float),
            "labels": torch.tensor(
                [b["labels"] for b in batch], dtype=torch.long),
        }


class BiEncoderClassifier(nn.Module):
    """Bi-encoder: shared XLM-R encoder for RFQ & Supplier,
    CLS pooling, element-wise diff, structured features \u2192 MLP head."""

    def __init__(self, model_name, num_labels, num_features, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size          # 1024 for large
        self.config  = self.encoder.config
        self.config.num_labels = num_labels

        self.head = nn.Sequential(
            nn.Linear(h * 3 + num_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_labels),
        )
        self._init_head()

    def _init_head(self):
        for m in self.head:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def _cls(self, input_ids, attention_mask):
        return self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state[:, 0]

    def forward(self, rfq_input_ids, rfq_attention_mask,
                sup_input_ids, sup_attention_mask,
                structured_features, labels=None, **kw):
        r = self._cls(rfq_input_ids, rfq_attention_mask)
        s = self._cls(sup_input_ids, sup_attention_mask)
        x = torch.cat([r, s, torch.abs(r - s),
                       structured_features], dim=-1)
        logits = self.head(x)
        return {"logits": logits}


model = BiEncoderClassifier(MODEL_NAME, NUM_LABELS, NUM_STRUCTURED)
total_p = sum(p.numel() for p in model.parameters())
print(f"Total parameters:       {total_p:,}")
print(f"Encoder hidden size:    {model.encoder.config.hidden_size}")

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters:       561,534,852
Encoder hidden size:    1024


In [23]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":    accuracy_score(labels, preds),
        "f1_macro":    f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }


class BiEncoderTrainer(Trainer):
    """Trainer subclass that routes bi-encoder inputs and uses Focal Loss."""

    def __init__(self, *args, focal_loss_fn=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss_fn = focal_loss_fn or FocalLoss()

    def _model_forward(self, model, inputs):
        return model(
            rfq_input_ids=inputs["rfq_input_ids"],
            rfq_attention_mask=inputs["rfq_attention_mask"],
            sup_input_ids=inputs["sup_input_ids"],
            sup_attention_mask=inputs["sup_attention_mask"],
            structured_features=inputs["structured_features"],
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels  = inputs["labels"]
        outputs = self._model_forward(model, inputs)
        loss    = self.focal_loss_fn(outputs["logits"], labels)
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only,
                        ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        with torch.no_grad():
            outputs = self._model_forward(model, inputs)
            logits  = outputs["logits"]
            loss    = self.focal_loss_fn(logits, inputs["labels"])
        if prediction_loss_only:
            return (loss, None, None)
        return (loss, logits, inputs["labels"])

## Gradual Unfreezing Training

| Phase | Trainable | LR | Epochs |
|-------|-----------|------|--------|
| 1 | Classifier head only | 1e-3 | 3 |
| 2 | Head + top 8 encoder layers | 2e-5 | 4 |
| 3 | All parameters | 5e-6 | 5 |

In [24]:
def freeze_encoder(m):
    for p in m.encoder.parameters():
        p.requires_grad = False

def unfreeze_top_n(m, n):
    for layer in m.encoder.encoder.layer[-n:]:
        for p in layer.parameters():
            p.requires_grad = True

def unfreeze_all(m):
    for p in m.parameters():
        p.requires_grad = True

def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


collator   = BiEncoderCollator(tokenizer.pad_token_id)
focal_loss = FocalLoss(gamma=2.0, label_smoothing=0.1,
                       num_classes=NUM_LABELS)
OUTPUT_DIR = "/content/xlmr-biencoder"

print("Utilities ready")

Utilities ready


In [25]:
print("=" * 60)
print("PHASE 1: Classifier head only")
print("=" * 60)

freeze_encoder(model)
model.encoder.gradient_checkpointing_disable()
print(f"Trainable parameters: {count_trainable(model):,}")

args_p1 = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/phase1",
    num_train_epochs=6,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=1,
    learning_rate=1e-3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=2,
    seed=42
)

trainer_p1 = BiEncoderTrainer(
    model=model,
    args=args_p1,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    compute_metrics=compute_metrics,
    focal_loss_fn=focal_loss,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer_p1.train()
p1_f1 = trainer_p1.state.best_metric
print(f"\nPhase 1 best val f1_macro: {p1_f1:.4f}")

PHASE 1: Classifier head only
Trainable parameters: 1,644,420


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.763430,0.754348,0.365443,0.261511,0.288507
2,0.754447,0.748623,0.364118,0.257426,0.282287
3,0.734894,0.754120,0.357494,0.294993,0.324385
4,0.711854,0.753276,0.360522,0.296214,0.323763
5,0.708838,0.754701,0.367714,0.306908,0.332548


KeyboardInterrupt: 

In [ ]:
print("=" * 60)
print("PHASE 2: Head + top 8 encoder layers")
print("=" * 60)

unfreeze_top_n(model, 8)
model.encoder.gradient_checkpointing_enable()
print(f"Trainable parameters: {count_trainable(model):,}")

args_p2 = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/phase2",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    weight_decay=0.1,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    fp16=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    dataloader_num_workers=2,
    seed=42
)

trainer_p2 = BiEncoderTrainer(
    model=model,
    args=args_p2,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    compute_metrics=compute_metrics,
    focal_loss_fn=focal_loss,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer_p2.train()
p2_f1 = trainer_p2.state.best_metric
print(f"\nPhase 2 best val f1_macro: {p2_f1:.4f}")

In [ ]:
print("=" * 60)
print("PHASE 3: Full fine-tuning")
print("=" * 60)

unfreeze_all(model)
model.encoder.gradient_checkpointing_enable()
print(f"Trainable parameters: {count_trainable(model):,}")

args_p3 = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/phase3",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=16,
    learning_rate=5e-6,
    weight_decay=0.1,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    fp16=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    dataloader_num_workers=2,
    seed=42,
    save_safetensors=False,
)

trainer_p3 = BiEncoderTrainer(
    model=model,
    args=args_p3,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    compute_metrics=compute_metrics,
    focal_loss_fn=focal_loss,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

trainer_p3.train()
p3_f1 = trainer_p3.state.best_metric
print(f"\nPhase 3 best val f1_macro: {p3_f1:.4f}")

## Test Set Evaluation

In [ ]:
test_out    = trainer_p3.predict(ds_test)
test_preds  = np.argmax(test_out.predictions, axis=-1)
test_labels = test_out.label_ids

label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]

print(classification_report(test_labels, test_preds,
                            target_names=label_names, digits=4))

cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("XLM-R Large Bi-Encoder \u2014 Test Confusion Matrix")
plt.tight_layout()
plt.show()

test_f1  = f1_score(test_labels, test_preds, average="macro")
test_acc = accuracy_score(test_labels, test_preds)
print(f"\nTest macro-F1:  {test_f1:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"\nPhase summary: P1={p1_f1:.4f}  P2={p2_f1:.4f}  P3={p3_f1:.4f}")

In [ ]:
SAVE_DIR = "/content/xlmr-biencoder-final"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "model_config": {
        "model_name": MODEL_NAME,
        "num_labels": NUM_LABELS,
        "num_features": NUM_STRUCTURED,
    },
}, os.path.join(SAVE_DIR, "bi_encoder.pt"))

tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to {SAVE_DIR}")
print(f"Contents: {', '.join(os.listdir(SAVE_DIR))}")

In [ ]:
DRIVE_DEST = "/content/drive/MyDrive/xlmr-biencoder-final"
shutil.copytree(SAVE_DIR, DRIVE_DEST, dirs_exist_ok=True)
print(f"Copied to {DRIVE_DEST}")

In [ ]:
test_cases = [
    {
        "rfq_title": "Korktaschen, Korkrucks\u00e4cke, Korkreisetaschen",
        "rfq_description": "Ich bin auf der Suche nach Korktaschen. Anzahl: 10-50, Einmalig.",
        "delivery_location": "DE",
        "quantity": "10 - 50",
        "rfq_supplier_types": "MANUFACTURER, WHOLESALER",
        "supplier_name": "FARL CORK UNIPESSOAL, LDA",
        "supplier_country": "PT",
        "distribution_area": "international",
        "supplier_description": "FARL CORK is a company in the cork industry, from forest to final product.",
        "supplier_types": "Production",
        "products": "Natural yoga block, Cork Specialties, Granules, Corks",
        "product_categories": "Wood and wood products",
        "keywords": "cork, yoga, granules",
    },
    {
        "rfq_title": "High-End Leather Card Holders / Wallets",
        "rfq_description": "Looking for manufacturing partner for card holder models. Premium quality.",
        "delivery_location": "FR",
        "quantity": "20-100",
        "rfq_supplier_types": "PRODUCTION",
        "supplier_name": "GUARNICIONERIA HNOS. PEDRAZA",
        "supplier_country": "ES",
        "distribution_area": "",
        "supplier_description": "WE ARE LEATHER ARTISANS, MANUFACTURING ALL TYPES OF ITEMS.",
        "supplier_types": "Production",
        "products": "",
        "product_categories": "Textile production",
        "keywords": "",
    },
    {
        "rfq_title": "Korktaschen, Korkrucks\u00e4cke, Korkreisetaschen",
        "rfq_description": "Ich bin auf der Suche nach Korktaschen. Anzahl: 10-50, Einmalig.",
        "delivery_location": "DE",
        "quantity": "10 - 50",
        "rfq_supplier_types": "MANUFACTURER, WHOLESALER",
        "supplier_name": "PROMOEASO",
        "supplier_country": "ES",
        "distribution_area": "europe",
        "supplier_description": "Online catalog with 5000 references for corporate and promotional gifts.",
        "supplier_types": "Wholesaler",
        "products": "Shopping Bags, promotional umbrellas, Customized Mugs",
        "product_categories": "Advertising materials",
        "keywords": "Regalos, bolsas, mochilas",
    },
    {
        "rfq_title": "High-quality No-stick Inner Coating Hot Pot Set",
        "rfq_description": "Interested in Hot Pot Set. Material: Stainless Steel.",
        "delivery_location": "DE",
        "quantity": "30173",
        "rfq_supplier_types": "PRODUCTION",
        "supplier_name": "CONZEPT Container Modulbau & Handel GmbH",
        "supplier_country": "AT",
        "distribution_area": "international",
        "supplier_description": "",
        "supplier_types": "Production",
        "products": "Container f\u00fcr Schulen, B\u00fcrocontainer, Sanit\u00e4rcontainer",
        "product_categories": "Beh\u00e4lter, Logistik",
        "keywords": "Kindergarten, Container, Modulbau",
    },
]

model.eval()
model.to(device)

for i, case in enumerate(test_cases):
    case = {k: str(v) if v is not None else "" for k, v in case.items()}
    row = pd.Series(case)
    rfq_text = build_rfq_text(row)
    sup_text = build_supplier_text(row)
    feats    = extract_features(row)

    rfq_enc = tokenizer(rfq_text, truncation=True,
                        max_length=RFQ_MAX_LEN,
                        return_tensors="pt").to(device)
    sup_enc = tokenizer(sup_text, truncation=True,
                        max_length=SUP_MAX_LEN,
                        return_tensors="pt").to(device)
    feat_t  = torch.tensor([feats], dtype=torch.float, device=device)

    with torch.no_grad():
        out = model(
            rfq_input_ids=rfq_enc["input_ids"],
            rfq_attention_mask=rfq_enc["attention_mask"],
            sup_input_ids=sup_enc["input_ids"],
            sup_attention_mask=sup_enc["attention_mask"],
            structured_features=feat_t,
        )
    probs  = F.softmax(out["logits"], dim=-1)[0]
    ranked = probs.argsort(descending=True)

    print(f"\n--- Test Case {i+1}: {case['rfq_title'][:50]} ---")
    print(f"    Supplier: {case['supplier_name']}")
    print(f"    Features: country={feats[0]:.0f}  region={feats[1]:.1f}  "
          f"type_j={feats[2]:.2f}")
    print("    Prediction:")
    for idx in ranked:
        lbl  = ID2LABEL[idx.item()]
        prob = probs[idx].item()
        print(f"      {lbl:>12s}: {prob:.4f}")